# RDD ETL Process — State and City Extraction

## Overview

This is a small **RDD-based ETL process** that takes source records containing both state and city information, splits them, separates state and city records, cleans them, and produces two transformed RDDs.

Example source record:

```text
State=AP<xml>City=Amaravati
```

The ETL flow is:

```text
Source RDD
    ↓
flatMap()
    ↓
Split State and City
    ↓
filter()
    ↓
Separate State / City
    ↓
map()
    ↓
Clean State / City
    ↓
Transformed RDDs
```

---

# 1. Source Data

The source RDD contains records like:

```text
State=AP<xml>City=Amaravati
State=AR<xml>City=Itanagar
State=AS<xml>City=Dispur
State=BR<xml>City=Patna
State=CG<xml>City=Raipur
```

Each record contains:

```text
State information
      +
<xml> delimiter
      +
City information
```

---

# 2. Extract — `flatMap()`

```python
list_split = src_rdd.flatMap(lambda x: x.split("<xml>"))
```

For example:

```text
State=AP<xml>City=Amaravati
```

Using:

```python
x.split("<xml>")
```

produces:

```text
["State=AP", "City=Amaravati"]
```

Because `flatMap()` is used, these become separate RDD elements.

For example:

```text
State=AP<xml>City=Amaravati
State=AR<xml>City=Itanagar
```

becomes approximately:

```text
State=AP
City=Amaravati
State=AR
City=Itanagar
```

## Why `flatMap()` instead of `map()`?

Using `map()`:

```python
src_rdd.map(lambda x: x.split("<xml>"))
```

would produce:

```text
["State=AP", "City=Amaravati"]
["State=AR", "City=Itanagar"]
```

Using `flatMap()`:

```python
src_rdd.flatMap(lambda x: x.split("<xml>"))
```

produces:

```text
State=AP
City=Amaravati
State=AR
City=Itanagar
```

### Key Point

> **`flatMap()` is used when one input record can produce multiple output elements and we want those elements flattened into the resulting RDD.**

---

# 3. Transform — Separate State and City

## State Records

```python
state_rdd = list_split.filter(
    lambda x: x.startswith("State=")
)
```

This keeps only records beginning with:

```text
State=
```

Result:

```text
State=AP
State=AR
State=AS
State=BR
State=CG
...
```

## City Records

```python
city_rdd = list_split.filter(
    lambda x: x.startswith("City=")
)
```

This keeps only records beginning with:

```text
City=
```

Result:

```text
City=Amaravati
City=Itanagar
City=Dispur
City=Patna
City=Raipur
...
```

Conceptually:

```text
list_split
    │
    ├── filter(State=)
    │       ↓
    │   state_rdd
    │
    └── filter(City=)
            ↓
        city_rdd
```

---

# 4. Transform — Clean State Data

Original data:

```text
State=AP
```

Code:

```python
state = state_rdd.map(
    lambda x: x.lower().replace("state=", "").upper()
)
```

The transformation happens in three steps.

### Step 1 — `lower()`

```text
State=AP
    ↓
state=ap
```

### Step 2 — `replace()`

```text
state=ap
    ↓
ap
```

### Step 3 — `upper()`

```text
ap
 ↓
AP
```

Final result:

```text
AP
```

Example:

```python
print(state.collect())
```

Output:

```text
['AP', 'AR', 'AS', 'BR', 'CG', ...]
```

---

# 5. Transform — Clean City Data

Original data:

```text
City=Amaravati
```

Code:

```python
city = city_rdd.map(
    lambda x: x.lower().replace("city=", "").title()
)
```

The transformation happens in three steps.

### Step 1 — `lower()`

```text
City=Amaravati
       ↓
city=amaravati
```

### Step 2 — `replace()`

```text
city=amaravati
       ↓
amaravati
```

### Step 3 — `title()`

```text
amaravati
     ↓
Amaravati
```

Final result:

```text
Amaravati
```

Example:

```python
print(city.collect())
```

Output:

```text
['Amaravati', 'Itanagar', 'Dispur', 'Patna', ...]
```

---

# 6. Complete ETL Code

```python
# Source RDD
src_rdd = sc.parallelize([
    "State=AP<xml>City=Amaravati",
    "State=AR<xml>City=Itanagar",
    "State=AS<xml>City=Dispur",
    "State=BR<xml>City=Patna",
    "State=CG<xml>City=Raipur",
    "State=GA<xml>City=Panaji",
    "State=GJ<xml>City=Gandhinagar",
    "State=HR<xml>City=Chandigarh",
    "State=HP<xml>City=Shimla",
    "State=JH<xml>City=Ranchi",
    "State=KA<xml>City=Bengaluru",
    "State=KL<xml>City=Thiruvananthapuram",
    "State=MP<xml>City=Bhopal",
    "State=MH<xml>City=Mumbai",
    "State=MN<xml>City=Imphal",
    "State=ML<xml>City=Shillong",
    "State=MZ<xml>City=Aizawl",
    "State=NL<xml>City=Kohima",
    "State=OD<xml>City=Bhubaneswar",
    "State=PB<xml>City=Chandigarh",
    "State=RJ<xml>City=Jaipur",
    "State=SK<xml>City=Gangtok",
    "State=TN<xml>City=Chennai",
    "State=TS<xml>City=Hyderabad",
    "State=TR<xml>City=Agartala",
    "State=UP<xml>City=Lucknow",
    "State=UK<xml>City=Dehradun",
    "State=WB<xml>City=Kolkata"
])

# Extract
list_split = src_rdd.flatMap(lambda x: x.split("<xml>"))

# Transform - separate State and City
state_rdd = list_split.filter(lambda x: x.startswith("State="))
city_rdd = list_split.filter(lambda x: x.startswith("City="))

# Transform - clean State
state = state_rdd.map(lambda x: x.lower().replace("state=", "").upper())

# Transform - clean City
city = city_rdd.map(lambda x: x.lower().replace("city=", "").title())

# View results
print(state.collect())
print(city.collect())
```

---

# 7. Simplified Cleaning Version

Since the source data consistently contains `State=` and `City=`, the `lower()` operation is not necessary before removing the prefix.

The transformations can be simplified to:

```python
state = state_rdd.map(
    lambda x: x.replace("State=", "").upper()
)

city = city_rdd.map(
    lambda x: x.replace("City=", "").title()
)
```

This produces the same result for the current source data.

---

# 8. ETL Mapping

| ETL Step | RDD Operation |
|---|---|
| Extract | `src_rdd` |
| Split source records | `flatMap()` |
| Separate State records | `filter()` |
| Separate City records | `filter()` |
| Clean State | `map()` |
| Clean City | `map()` |
| View result | `collect()` |

### Important

The current program performs:

```text
Extract + Transform
```

It does **not yet contain a real Load step**.

`collect()` only brings the transformed results to the Driver for viewing. It should not be considered a production data-loading operation.

---

# 9. Complete Pipeline

```text
                         SOURCE
                           │
                           ↓
                        src_rdd
                           │
                           ↓
                     flatMap(split)
                           │
                           ↓
                       list_split
                           │
                  ┌────────┴────────┐
                  ↓                 ↓
            filter(State=)    filter(City=)
                  ↓                 ↓
              state_rdd          city_rdd
                  ↓                 ↓
              map(clean)        map(clean)
                  ↓                 ↓
                state              city
```

---

# 10. RDD Concepts Used

This ETL example combines the following RDD operations:

```text
flatMap()
   ↓
filter()
   ↓
map()
   ↓
collect()
```

### `flatMap()`

Used to split each source record into multiple RDD elements.

```python
src_rdd.flatMap(lambda x: x.split("<xml>"))
```

### `filter()`

Used to select records based on a condition.

```python
list_split.filter(lambda x: x.startswith("State="))
```

### `map()`

Used to transform each element.

```python
state_rdd.map(lambda x: x.replace("State=", "").upper())
```

### `collect()`

Used to bring the RDD results back to the Driver.

```python
state.collect()
```

---

# Key Learning

This is a simple example of how RDD transformations can be combined to perform an ETL workflow:

```text
Source Data
    ↓
Extract
    ↓
flatMap()
    ↓
Filter / Separate
    ↓
Transform / Clean
    ↓
Final RDD
```

The important pattern to remember is:

```text
flatMap() → Split
filter()  → Select
map()     → Transform
collect() → Retrieve result
```
